# Safety Guard LoRA - Gemma 4 12B

Purpose: train a Nemotron-style content safety classifier LoRA for OpenWebUI content safety and policy filters.

This notebook is intentionally scoped to content safety. It uses Nemotron Safety Guard / Aegis-style safe and unsafe examples and excludes jailbreak-specific samples from the primary safety training mix. Prompt injection gets its own notebook and output contract.

Ported from `safety_guard_lora_qwen3_14b.ipynb` — same dataset curation, taxonomy, and output contract, just on the Gemma 4 12B base (Gemma 4 support requires `transformers`/`peft` from git main; see the Environment Preparation cell below). Serves as a LoRA on the same A5000 alongside the other safety/persona LoRAs.

Matching filters:
- `openwebui-safety-filters/content_safety/filter/safety_guard_filter_v3.py`
- `openwebui-safety-filters/policy_violation/filter/safety_filter_company_policy_violation_v1.py`

Expected model output:
```json
{"User Safety": "safe|unsafe", "Response Safety": "safe|unsafe", "Safety Categories": "..."}
```

In [ ]:
# Environment preparation
# Install core packages in the running notebook container
!pip install -q -U unsloth trl accelerate datasets bitsandbytes

# Container ships torchao 0.14.0+git (custom aarch64 build). peft requires
# torchao>=0.16.0 OR torchao absent. No aarch64 wheel ≥0.16 on PyPI, so
# uninstall — peft's torchao dispatcher then no-ops and falls through to
# the bnb 4-bit dispatcher, which is what we want for QLoRA anyway.
!pip uninstall -y -q torchao

# Gemma 4 support may be ahead of PyPI releases.
!pip install -q -U git+https://github.com/huggingface/transformers.git

# Keep PEFT compatible with latest Transformers main.
!pip install -q -U git+https://github.com/huggingface/peft.git

# Verify installations
import importlib.util
import unsloth
import transformers
import peft
import trl
print(f"✓ Unsloth: {unsloth.__version__}")
print(f"✓ Transformers: {transformers.__version__}")
print(f"✓ PEFT: {peft.__version__}")
print(f"✓ TRL: {trl.__version__}")
print(f"✓ torchao installed: {importlib.util.find_spec('torchao') is not None} (should be False)")
print("Environment ready. Restart kernel, then rerun from Cell 3 (Configuration).")

In [ ]:
# Configuration
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_ROOT = PROJECT_ROOT / 'output'
# unsloth/gemma-4-12b-it has no dedicated pre-quantized bnb-4bit repo on HF;
# Unsloth applies bnb 4-bit quantization on the fly via load_in_4bit=True below.
BASE_LLM = 'unsloth/gemma-4-12b-it'
MODEL_NAME_BASE = 'safety_guard_gemma4_12b_content_safety'
OUTPUT_BASE_DIR = OUTPUT_ROOT / MODEL_NAME_BASE
CHECKPOINT_DIR = OUTPUT_BASE_DIR / 'train'
LORA_OUTPUT_DIR = OUTPUT_BASE_DIR / 'lora_adapters'

NEMOTRON_DATASET = 'nvidia/Nemotron-Safety-Guard-Dataset-v3'
MAX_SEQ_LENGTH = 4096
SEED = 42

# Keep this LoRA focused on content safety, not prompt injection.
EXCLUDE_NEMOTRON_TAGS = {'jailbreaking'}
MAX_UNSAFE_PER_CATEGORY = 450
SAFE_RATIO_TO_UNSAFE = 0.82

BATCH_SIZE = 2
GRAD_ACCUM = 8
LEARNING_RATE = 5e-5
TARGET_EPOCHS = 1
WARMUP_STEPS = 50

LORA_R = 32
LORA_ALPHA = 32
LORA_DROPOUT = 0
LORA_TARGET_MODULES = [
    'q_proj', 'k_proj', 'v_proj', 'o_proj',
    'gate_proj', 'up_proj', 'down_proj',
]

print(f'Base model: {BASE_LLM}')
print(f'Output: {LORA_OUTPUT_DIR}')

In [ ]:
# Imports
import os

# DGX Spark (sm_120 / GB10): disable Unsloth's flex_attention override and broken
# torch.compile path. Without these, Gemma falls back to slow eager Python loops.
# Must be set BEFORE `import unsloth`.
os.environ["UNSLOTH_ENABLE_FLEX_ATTENTION"] = "0"
os.environ["UNSLOTH_COMPILE_DISABLE"] = "1"

import json
import random
import collections
from datasets import load_dataset, concatenate_datasets
from unsloth import FastLanguageModel
from trl import SFTTrainer, SFTConfig

random.seed(SEED)

In [ ]:
# Nemotron / Aegis taxonomy and classifier prompt
SAFETY_TAXONOMY = '''S1: Violence.
S2: Sexual.
S3: Criminal Planning/Confessions.
S4: Guns and Illegal Weapons.
S5: Controlled/Regulated Substances.
S6: Suicide and Self Harm.
S7: Sexual (minor).
S8: Hate/Identity Hate.
S9: PII/Privacy.
S10: Harassment.
S11: Threat.
S12: Profanity.
S13: Needs Caution.
S14: Other.
S15: Manipulation.
S16: Fraud/Deception.
S17: Malware.
S18: High Risk Gov Decision Making.
S19: Political/Misinformation/Conspiracy.
S20: Copyright/Trademark/Plagiarism.
S21: Unauthorized Advice.
S22: Illegal Activity.
S23: Immoral/Unethical.'''

CLASSIFIER_INSTRUCTION = f'''Task: Check if there is unsafe content in the user or agent messages in the below conversation in accordance with our safety policy with the below categories. Provide a 'safe' or 'unsafe' rating for both the user message and the agent response. If there is unsafe content, please also provide a list of violated categories according to our safety policy below.

<BEGIN UNSAFE CONTENT CATEGORIES>
{SAFETY_TAXONOMY}
<END UNSAFE CONTENT CATEGORIES>'''

CATEGORY_MAP = {
    'Violence': 'Violence',
    'Sexual': 'Sexual',
    'Criminal Planning/Confessions': 'Criminal Planning/Confessions',
    'Guns and Illegal Weapons': 'Guns and Illegal Weapons',
    'Controlled/Regulated Substances': 'Controlled/Regulated Substances',
    'Suicide and Self Harm': 'Suicide and Self Harm',
    'Sexual (minor)': 'Sexual (minor)',
    'Hate/Identity Hate': 'Hate/Identity Hate',
    'PII/Privacy': 'PII/Privacy',
    'Harassment': 'Harassment',
    'Threat': 'Threat',
    'Profanity': 'Profanity',
    'Needs Caution': 'Needs Caution',
    'Other': 'Other',
    'Manipulation': 'Manipulation',
    'Fraud/Deception': 'Fraud/Deception',
    'Malware': 'Malware',
    'High Risk Gov Decision Making': 'High Risk Gov Decision Making',
    'Political/Misinformation/Conspiracy': 'Political/Misinformation/Conspiracy',
    'Copyright/Trademark/Plagiarism': 'Copyright/Trademark/Plagiarism',
    'Unauthorized Advice': 'Unauthorized Advice',
    'Illegal Activity': 'Illegal Activity',
    'Immoral/Unethical': 'Immoral/Unethical',
}

In [ ]:
# Load and curate Nemotron content-safety data
raw = load_dataset(NEMOTRON_DATASET, split='train')

def keep_content_safety(example):
    if not example.get('prompt') or example.get('prompt') == 'REDACTED':
        return False
    if example.get('language') not in (None, 'en'):
        return False
    if example.get('tag') in EXCLUDE_NEMOTRON_TAGS:
        return False
    return True

dataset = raw.filter(keep_content_safety)
unsafe_pool = dataset.filter(lambda x: x.get('prompt_label') == 'unsafe' or x.get('response_label') == 'unsafe')
safe_pool = dataset.filter(lambda x: x.get('prompt_label') == 'safe' and (not x.get('response_label') or x.get('response_label') == 'safe'))

category_groups = collections.defaultdict(list)
for idx, example in enumerate(unsafe_pool):
    cats = [c.strip() for c in str(example.get('violated_categories', '')).split(',') if c.strip()]
    key = cats[0] if cats else 'Other'
    category_groups[key].append(idx)

selected_unsafe = []
for indices in category_groups.values():
    random.shuffle(indices)
    selected_unsafe.extend(indices[:MAX_UNSAFE_PER_CATEGORY])

nemotron_unsafe = unsafe_pool.select(selected_unsafe)
safe_count = min(len(safe_pool), int(len(nemotron_unsafe) * SAFE_RATIO_TO_UNSAFE))
safe_indices = list(range(len(safe_pool)))
random.shuffle(safe_indices)
nemotron_safe = safe_pool.select(safe_indices[:safe_count])

print(f'Unsafe content-safety examples: {len(nemotron_unsafe)}')
print(f'Safe examples: {len(nemotron_safe)}')
print(f'Excluded tags: {EXCLUDE_NEMOTRON_TAGS}')

In [ ]:
# Format examples into the OpenWebUI Safety Guard contract
def normalize_categories(value):
    labels = []
    for raw_cat in str(value or '').split(','):
        cat = raw_cat.strip()
        if not cat:
            continue
        labels.append(CATEGORY_MAP.get(cat, cat))
    return ','.join(dict.fromkeys(labels))

def build_classifier_prompt(prompt_text, response_text=None):
    conversation = f'\nuser: {prompt_text}'
    if response_text:
        conversation += f'\nresponse: agent: {response_text}'
    return f'''{CLASSIFIER_INSTRUCTION}

<BEGIN CONVERSATION>
{conversation}
<END CONVERSATION>

Provide your safety assessment for the above conversation in the below output JSON format:
{{"User Safety": safe or unsafe rating for the user message, "Response Safety": safe or unsafe rating for the agent response. Omit if no agent response present. "Safety Categories": a comma-separated list of applicable safety categories from the provided taxonomy. Omit if all safe.}}

Do not include anything other than the output JSON in your response.
Output JSON: '''

def build_classifier_output(prompt_label, response_label=None, categories=''):
    result = {'User Safety': prompt_label or 'safe'}
    if response_label:
        result['Response Safety'] = response_label
    if categories:
        result['Safety Categories'] = categories
    return json.dumps(result, ensure_ascii=False)

def format_example(example):
    user_text = example.get('prompt') or ''
    response_text = example.get('response') or None
    categories = normalize_categories(example.get('violated_categories', ''))
    messages = [
        {'role': 'user', 'content': build_classifier_prompt(user_text, response_text)},
        {'role': 'assistant', 'content': build_classifier_output(example.get('prompt_label'), example.get('response_label'), categories)},
    ]
    return {'text': tokenizer.apply_chat_template(messages, tokenize=False)}

In [ ]:
# Load base model and tokenizer
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_LLM,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,
)

# Gemma 4 can return a Processor object instead of a plain tokenizer.
if hasattr(tokenizer, "tokenizer"):
    tokenizer = tokenizer.tokenizer

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

train_dataset = concatenate_datasets([nemotron_unsafe, nemotron_safe]).shuffle(seed=SEED)
train_dataset = train_dataset.map(format_example, remove_columns=train_dataset.column_names)
train_dataset = train_dataset.filter(lambda x: 100 < len(x['text']) <= MAX_SEQ_LENGTH * 4)
split = train_dataset.train_test_split(test_size=0.05, seed=SEED)

print(split)

In [ ]:
# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
)

In [ ]:
# Train
import torch

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=split['train'],
    eval_dataset=split['test'],
    args=SFTConfig(
        dataset_text_field='text',
        max_length=MAX_SEQ_LENGTH,
        packing=False,
        output_dir=str(CHECKPOINT_DIR),
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=TARGET_EPOCHS,
        learning_rate=LEARNING_RATE,
        warmup_steps=WARMUP_STEPS,
        lr_scheduler_type='cosine',
        logging_steps=10,
        eval_strategy='steps',
        eval_steps=100,
        save_steps=200,
        save_total_limit=3,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        optim='adamw_8bit',
        seed=SEED,
        report_to='none',
    ),
)

trainer.train()

In [ ]:
# Save adapter and training metadata
LORA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(LORA_OUTPUT_DIR))
tokenizer.save_pretrained(str(LORA_OUTPUT_DIR))

metadata = {
    'purpose': 'content_safety',
    'base_model': BASE_LLM,
    'output_contract': 'Nemotron JSON safety classifier',
    'matching_filters': [
        'content_safety/filter/safety_guard_filter_v3.py',
        'policy_violation/filter/safety_filter_company_policy_violation_v1.py',
    ],
    'datasets': [NEMOTRON_DATASET],
    'excluded_tags': sorted(EXCLUDE_NEMOTRON_TAGS),
    'lora': {
        'r': LORA_R,
        'alpha': LORA_ALPHA,
        'target_modules': LORA_TARGET_MODULES,
    },
}
with open(LORA_OUTPUT_DIR / 'safety_guard_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Saved Safety Guard LoRA to {LORA_OUTPUT_DIR}')